In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [6]:
corpus = [
    "machine learning is a field of artificial intelligence",
    "deep learning is a subset of machine learning",
    "machine learning models learn from data",
    "deep learning models use neural networks",
    "artificial intelligence enables machines to think",
    "neural networks are inspired by the human brain",
    "data science uses machine learning techniques",
    "machine learning and deep learning are related fields"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index)+1
print("Total Words:",total_words)

Total Words: 34


In [7]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
input_sequences = []
for line in corpus:
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)):
    input_sequences.append(token_list[:i+1])
max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen = max_len, padding = 'pre')
X  = input_sequences[:,:-1]
y = input_sequences[:,-1]

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
model = Sequential([
    Embedding(total_words, 50, input_length = max_len-1),
    SimpleRNN(100),
    Dense(total_words,activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=["accuracy"])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.fit(X,y,epochs=200,verbose=1)

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.0243 - loss: 3.5315
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2118 - loss: 3.4652
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.2222 - loss: 3.4025
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.1875 - loss: 3.3424
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.1979 - loss: 3.2615
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.1736 - loss: 3.2007
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.1632 - loss: 3.1527 
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.1736 - loss: 3.0762 
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.1632 - loss: 3.0836
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.2118 - loss: 3.0101
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.2118 - loss: 2.9448
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.2535 - 

In [10]:
def predict_next_word(model,tokenizer,text,max_len):
  token_list = tokenizer.texts_to_sequences([text])[0]
  token_list = pad_sequences([token_list],maxlen=max_len-1,padding='pre')
  probs = model.predict(token_list,verbose=0)
  predicted_index = np.argmax(probs)
  for word,index in tokenizer.word_index.items():
    if index == predicted_index:
      return word

In [11]:
seed_text = "Machine Learning"
print("Next Word:",predict_next_word(model,tokenizer,seed_text,max_len))

Next Word: is


In [12]:
def generate_text(seed_text,n_words):
  for _ in range(n_words):
    next_word = predict_next_word(model,tokenizer,seed_text,max_len)
    seed_text += " "+next_word
  return seed_text
print(generate_text("Machine learning",6))

Machine learning is a field of artificial intelligence


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
model = Sequential([
    Embedding(total_words,50,input_length=max_len-1),
    LSTM(100),
    Dense(total_words,activation="softmax")
])
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.fit(X, y, epochs=200, verbose=1)

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.0625 - loss: 3.5256
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1528 - loss: 3.5173
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1424 - loss: 3.5095
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.1632 - loss: 3.4986
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.1528 - loss: 3.4882
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1424 - loss: 3.4745
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.1736 - loss: 3.4505
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.1736 - loss: 3.4255
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1632 - loss: 3.3887
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.1632 - loss: 3.3384
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.1424 - loss: 3.2909
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.1528 - lo

In [15]:
def predict_next_word(model, tokenizer, text, max_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted)

    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word

In [16]:
seed_text = "machine learning"
next_word = predict_next_word(model, tokenizer, seed_text, max_len)
print("Next word:", next_word)

Next word: models


In [17]:
def generate_text(seed_text, n_words):
    for _ in range(n_words):
        next_word = predict_next_word(model, tokenizer, seed_text, max_len)
        seed_text += " " + next_word
    return seed_text

print(generate_text("machine learning", 5))

machine learning models learn from data fields


In [21]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
def predict_next_word_top_k(model,tokenizer,text,max_len,k=3):
  token_list = tokenizer.texts_to_sequences([text])[0]
  token_list = pad_sequences([token_list],maxlen=max_len-1,padding='pre')
  probs = model.predict(token_list,verbose=0)[0]
  top_k_indices = np.argsort(probs)[-k:]

  top_k_probs = probs[top_k_indices]
  top_k_probs = top_k_probs/np.sum(top_k_probs)
  predicted_index = np.random.choice(top_k_indices,p=top_k_probs)
  for word,index in tokenizer.word_index.items():
    if index == predicted_index:
      return word


In [22]:
def generate_text_top_k(seed_text,n_words,k=3):
  for _ in range(n_words):
    next_word = predict_next_word_top_k(model,tokenizer,seed_text,max_len,k)
    seed_text += " "+next_word
  return seed_text

In [23]:
print(generate_text_top_k("machine learning",8,k=3))

machine learning models learn from data fields fields brain brain


In [24]:
def sample_next_word(model,tokenizer,text,max_len,k=5,temperature=1.0):
  token_list = tokenizer.texts_to_sequences([text])[0]
  token_list = pad_sequences([token_list],maxlen=max_len-1,padding="pre")

  probs = model.predict(token_list,verbose=0)[0]

  probs = np.log(probs+1e-9)/temperature
  probs = np.exp(probs)
  probs = probs/np.sum(probs)

  top_k_indices = np.argsort(probs)[-k:]
  top_k_probs = probs[top_k_indices]
  top_k_probs = top_k_probs/np.sum(top_k_probs)

  predicted_index = np.random.choice(top_k_indices,p=top_k_probs)
  for word,index in tokenizer.word_index.items():
    if index == predicted_index:
      return word

In [25]:
def generate_text(
    seed_text,
    n_words,
    model,
    tokenizer,
    max_len,
    k=5,
    temperature=1.0
):
    for _ in range(n_words):
        next_word = sample_next_word(
            model,
            tokenizer,
            seed_text,
            max_len,
            k,
            temperature
        )
        seed_text += " " + next_word
    return seed_text


In [26]:
print("Low Temp:")
print(generate_text("machine learning", 8, model, tokenizer, max_len,
                    k=5, temperature=0.5))

print("\nMedium Temp:")
print(generate_text("machine learning", 8, model, tokenizer, max_len,
                    k=5, temperature=1.0))

print("\nHigh Temp:")
print(generate_text("machine learning", 8, model, tokenizer, max_len,
                    k=5, temperature=1.5))

Low Temp:
machine learning models learn from data data fields brain brain

Medium Temp:
machine learning models learn from data data fields fields brain

High Temp:
machine learning is deep learning are related fields brain human
